# 🚀 Spaceship Titanic
## Predicting which passengers are transported to an alternate dimension

Bienvenidos al año 2912, donde tus habilidades en ciencia de datos son necesarias para resolver un misterio cósmico. Hemos recibido una transmisión desde cuatro años luz de distancia y la situación no pinta bien.

La **Spaceship Titanic** era un transatlántico interestelar que, durante su viaje inaugural, colisionó con una anomalía espaciotemporal oculta en una nube de polvo cerca de Alfa Centauri. Aunque la nave permaneció intacta, casi la mitad de sus ~13.000 pasajeros fueron transportados a una dimensión alternativa.

**Tu misión:** predecir, a partir de los registros recuperados del sistema dañado, qué pasajeros fueron transportados. Cada predicción correcta puede significar un rescate.


---
## 📋 Diccionario de variables

| Variable | Descripción |
|---|---|
| **PassengerId** | ID único con formato `gggg_pp` (grupo + posición en el grupo) |
| **HomePlanet** | Planeta de origen del pasajero |
| **CryoSleep** | Si el pasajero eligió criosueño (quedó confinado en su camarote) |
| **Cabin** | Camarote en formato `deck/num/side` (P = babor, S = estribor) |
| **Destination** | Planeta de destino |
| **Age** | Edad del pasajero |
| **VIP** | Si pagó servicio VIP especial |
| **RoomService / FoodCourt / ShoppingMall / Spa / VRDeck** | Gasto facturado en cada servicio de lujo |
| **Name** | Nombre y apellido |
| **Transported** | 🎯 **Variable objetivo**: si fue transportado a otra dimensión (True/False) |


---

### 1️⃣ Importación de librerías y carga del dataset

Importo las librerías necesarias para el proyecto: **pandas** y **numpy** para manipular datos, **sklearn** para dividir en train/test, y **TensorFlow/Keras** para construir y entrenar la red neuronal.


In [ ]:
import pandas as pd          # manipulación y análisis de datos en tablas (DataFrames)
import numpy as np             # operaciones numéricas y manejo de arrays
import matplotlib.pyplot as plt  # visualización de gráficas
import seaborn as sns          # gráficas estadísticas de mayor nivel

from sklearn.model_selection import train_test_split  # divido mis datos en entrenamiento y test
from sklearn.metrics import classification_report, confusion_matrix  # métricas de evaluación

import tensorflow as tf                        # framework de deep learning
from tensorflow import keras                   # API de alto nivel para construir redes neuronales
from tensorflow.keras import models, layers   # modelos y capas de la red


In [ ]:
# cargo el archivo CSV en un DataFrame de pandas llamado 'df'
df = pd.read_csv('space_titanic.csv')


---
### 2️⃣ Exploración inicial del dataset (EDA)

Antes de tocar nada, observo la estructura del dataset: cuántas filas y columnas tiene, qué tipo de dato tiene cada columna y si hay valores nulos.


In [ ]:
# muestro un resumen estructural del dataframe:
# número de filas y columnas, tipo de dato de cada columna y cuántos valores no nulos hay
df.info()


In [ ]:
# calculo estadísticas descriptivas de las columnas numéricas:
# media, desviación estándar, mínimo, máximo y percentiles (25%, 50%, 75%)
df.describe()


In [ ]:
# visualizo las primeras 10 filas del dataset para hacerme una idea del aspecto real de los datos
df.head(10)


In [ ]:
# cuento el número de valores nulos (NaN) en cada columna
# un valor nulo significa que ese dato falta para ese pasajero
print("\nNulos por columna:")
print(df.isnull().sum())


In [ ]:
# compruebo si hay filas exactamente duplicadas en el dataset
# las filas duplicadas podrían distorsionar el entrenamiento del modelo
print("\nFilas Duplicadas:")
print(df.duplicated().sum())


---
### 3️⃣ Ingeniería de variables (Feature Engineering)

La columna `Cabin` almacena tres datos distintos juntos en un mismo string, con el formato `deck/num/side`. Al modelo le resulta mucho más útil tener esa información separada en tres columnas independientes.

- **Cabin_Deck**: letra de la cubierta (ej. `A`, `B`, `G`...)
- **Cabin_Num**: número de camarote (valor numérico)
- **Cabin_Side**: lado del barco (`P` = babor, `S` = estribor)

Una vez extraídas las tres partes, elimino la columna original `Cabin` porque ya no aporta información adicional.


In [ ]:
# separo la columna 'Cabin' en tres nuevas columnas usando '/' como separador del texto
# expand=True hace que cada parte del split acabe en una columna distinta
df[['Cabin_Deck', 'Cabin_Num', 'Cabin_Side']] = df['Cabin'].str.split('/', expand=True)

# elimino la columna 'Cabin' original porque ya no la necesito: toda su info está en las tres nuevas
df = df.drop(columns=['Cabin'])

# convierto 'Cabin_Num' de texto (object) a número (float) para poder usarla como variable numérica
df['Cabin_Num'] = pd.to_numeric(df['Cabin_Num'])

# reviso las primeras filas de las nuevas columnas para confirmar que la separación fue correcta
df[['Cabin_Deck', 'Cabin_Num', 'Cabin_Side']].head()


---
### 4️⃣ Tratamiento de valores nulos

Las redes neuronales no pueden trabajar con valores nulos (NaN): necesitan que todas las celdas tengan un valor numérico concreto.

**Estrategia de imputación:**
- **Variables numéricas** → relleno con la **mediana** (más robusta que la media cuando hay valores extremos)
- **Variables categóricas** → relleno con la **moda** (el valor más frecuente)
- **Columna `Name`** → la elimino directamente: el nombre de un pasajero no aporta información predictiva para el modelo


In [ ]:
# defino la lista de columnas numéricas que necesitan imputación
columnas_numericas = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Cabin_Num']

# recorro cada columna numérica y relleno sus nulos con la mediana de esa columna
for col in columnas_numericas:
    mediana = df[col].median()   # calculo la mediana de los valores existentes en esa columna
    df[col] = df[col].fillna(mediana)  # sustituyo los NaN por ese valor

# defino la lista de columnas categóricas (texto) que necesitan imputación
columnas_categoricas = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Cabin_Deck', 'Cabin_Side']

# recorro cada columna categórica y relleno sus nulos con la moda (valor más frecuente)
for col in columnas_categoricas:
    moda = df[col].mode()[0]     # mode() devuelve una serie, cojo el primer resultado con [0]
    df[col] = df[col].fillna(moda)  # sustituyo los NaN por ese valor

# elimino la columna 'Name' porque el nombre del pasajero no es una variable predictiva útil
df = df.drop(columns=['Name'])

# verifico que no queda ningún valor nulo en el dataset
df.isnull().sum()


---
### 5️⃣ Transformación de variables categóricas a numéricas

Las redes neuronales solo entienden números. Por eso convierto todas las variables no numéricas:

- **Booleanas** (`CryoSleep`, `VIP`): paso `True/False` → `1/0` directamente
- **Categóricas con múltiples clases** (`HomePlanet`, `Destination`, `Cabin_Deck`, `Cabin_Side`): aplico **One-Hot Encoding**, que crea una columna binaria (0/1) por cada categoría posible. Uso `drop_first=True` para evitar multicolinealidad (dummy variable trap)
- **Variable objetivo** (`Transported`): también la convierto a `1/0` pero de forma separada, porque no es una variable predictora


In [ ]:
# convierto las variables booleanas True/False a enteros 1/0
columnas_booleanas = ['CryoSleep', 'VIP']
for col in columnas_booleanas:
    df[col] = df[col].astype(int)  # astype(int) transforma True→1 y False→0

# defino las columnas categóricas de texto con múltiples categorías posibles
columnas_categoricas = ['HomePlanet', 'Destination', 'Cabin_Deck', 'Cabin_Side']

# aplico One-Hot Encoding: crea una columna 0/1 por cada categoría
# drop_first=True elimina la primera categoría de cada grupo para evitar redundancia lineal
df = pd.get_dummies(df, columns=columnas_categoricas, drop_first=True)

# convierto la variable objetivo 'Transported' a entero por separado, ya que no es predictora
df['Transported'] = df['Transported'].astype(int)

# muestro las primeras filas para confirmar que todo el dataset es ya completamente numérico
df.head()


---
### 6️⃣ División en conjuntos de entrenamiento y test

Separo los datos en dos bloques:
- **X_train / y_train (80%)**: los datos con los que la red aprende los patrones
- **X_test / y_test (20%)**: los datos que reservo para evaluar cómo funciona la red con ejemplos que nunca ha visto

Fijo `random_state=11` para que la división sea reproducible (siempre se obtiene el mismo resultado).


In [ ]:
# defino x como todas las columnas predictoras: elimino 'PassengerId' (es un ID, no predice nada)
# y también elimino 'Transported' que es la variable objetivo que quiero predecir
x = df.drop(columns=['PassengerId', 'Transported'])

# defino y como la variable objetivo: lo que el modelo tiene que aprender a predecir
y = df['Transported']

# divido en 80% entrenamiento y 20% test
# stratify=y asegura que la proporción de casos positivos/negativos se mantiene igual en ambos bloques
# random_state=11 fija la semilla para que la división sea siempre la misma
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=11, stratify=y)

# imprimo el tamaño de los bloques para comprobar que la división es correcta
print(f"filas para que mi red entrene: {x_train.shape[0]}")
print(f"filas para examinar a mi red: {x_test.shape[0]}")
print(f"variables predictoras: {x_train.shape[1]}")


---
### 7️⃣ Diseño y construcción de la red neuronal

Construyo una red neuronal densa (**Fully Connected / MLP**) con la siguiente arquitectura:

| Capa | Neuronas | Activación | Función |
|---|---|---|---|
| Entrada | `n_features` | — | Recibe los datos de entrada |
| Oculta 1 | 32 | ReLU | Detecta patrones de primer nivel |
| Oculta 2 | 16 | ReLU | Refina los patrones encontrados |
| Salida | 1 | Sigmoid | Devuelve una probabilidad entre 0 y 1 |

**¿Por qué sigmoid en la salida?** Porque es un problema de **clasificación binaria** (transportado: sí/no). La función sigmoid convierte cualquier valor en una probabilidad entre 0 y 1, que luego se interpreta como: > 0.5 → transportado, ≤ 0.5 → no transportado.


In [ ]:
# guardo el número de variables predictoras que tiene mi bloque de entrenamiento
# esto me permite definir el tamaño de la capa de entrada de forma dinámica
dimension_entrada = x_train.shape[1]

# construyo el modelo usando keras.Sequential: las capas se apilan una tras otra en orden
modelo_red = keras.Sequential([

    # defino el tamaño de la entrada usando el número de columnas de x_train
    # shape=(dimension_entrada,) le dice a la red cuántos valores recibe por pasajero
    keras.Input(shape=(dimension_entrada,)),

    # primera capa oculta con 32 neuronas y activación ReLU
    # ReLU (Rectified Linear Unit) activa la neurona si recibe una señal positiva, sino devuelve 0
    # con 32 neuronas la red puede detectar hasta 32 patrones distintos en los datos
    layers.Dense(32, activation='relu'),

    # segunda capa oculta con 16 neuronas y activación ReLU
    # al reducir de 32 a 16 neuronas, la red comprime y refina los patrones aprendidos
    layers.Dense(16, activation='relu'),

    # capa de salida con 1 sola neurona y activación sigmoid
    # sigmoid transforma el resultado en una probabilidad entre 0 y 1
    # probabilidad > 0.5 → el modelo predice que el pasajero fue transportado
    layers.Dense(1, activation='sigmoid')
])

# muestro un resumen de la arquitectura: capas, neuronas y parámetros entrenables
modelo_red.summary()


---
### 8️⃣ Compilación de la red neuronal

Antes de entrenar, configuro tres elementos clave:

- **Optimizer `adam`**: el algoritmo que ajusta los pesos durante el entrenamiento. Adam es una versión mejorada de descenso por gradiente, muy eficiente y ampliamente usado.
- **Loss `binary_crossentropy`**: la función de pérdida para problemas de clasificación binaria. Mide cuánto se equivoca el modelo en cada predicción.
- **Metric `accuracy`**: la métrica que monitorizo durante el entrenamiento para ver cómo evoluciona el porcentaje de aciertos.


In [ ]:
# compilo el modelo configurando el optimizador, la función de pérdida y la métrica de seguimiento
modelo_red.compile(
    optimizer='adam',           # adam ajusta automáticamente el ritmo de aprendizaje durante el entrenamiento
    loss='binary_crossentropy', # función de pérdida estándar para clasificación binaria (0 o 1)
    metrics=['accuracy']        # métrica de evaluación: porcentaje de predicciones correctas
)


---
### 9️⃣ Entrenamiento de la red neuronal

Entreno el modelo durante **20 epochs** con lotes de **32 muestras**. En cada epoch:
1. La red procesa todos los datos de entrenamiento en lotes de 32 pasajeros
2. Calcula cuánto se equivoca (loss)
3. Ajusta los pesos internos para equivocarse menos en la siguiente vuelta

Paso `validation_data=(x_test, y_test)` para ver en tiempo real cómo evoluciona el rendimiento sobre datos que el modelo **nunca ha visto**, y así detectar si hay sobreajuste (overfitting).


In [ ]:
# entreno la red neuronal y guardo el historial de métricas en cada epoch
historial = modelo_red.fit(
    x_train, y_train,          # datos y etiquetas de entrenamiento
    epochs=20,                 # número de veces que la red recorre todo el conjunto de entrenamiento
    batch_size=32,             # número de muestras que procesa la red antes de actualizar sus pesos
    validation_data=(x_test, y_test)  # datos de validación: evalúa el modelo al final de cada epoch
)


---
### 🔟 Evaluación del modelo

Una vez entrenado el modelo, evalúo su rendimiento de dos formas:

1. **Curvas de entrenamiento**: visualizo cómo evolucionan la pérdida y la precisión a lo largo de los epochs, tanto en train como en validación. Me ayudan a detectar sobreajuste (overfitting) o falta de ajuste (underfitting).

2. **Métricas finales sobre el conjunto de test**: accuracy, precisión, recall y F1-score para cada clase.


In [ ]:
# creo una figura con dos gráficas en paralelo para visualizar la evolución del entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- gráfica 1: evolución de la pérdida (loss) ---
# una pérdida de validación que sube mientras la de entrenamiento baja indica sobreajuste
ax1.plot(historial.history['loss'], label='loss entrenamiento', color='steelblue')         # pérdida en train
ax1.plot(historial.history['val_loss'], label='loss validación', color='tomato')           # pérdida en test
ax1.set_title('Curva de pérdida (Loss)')                 # título de la gráfica
ax1.set_xlabel('Epoch')                                   # eje x: número de época
ax1.set_ylabel('Loss (Binary Crossentropy)')              # eje y: valor de la pérdida
ax1.legend()                                              # muestro la leyenda

# --- gráfica 2: evolución de la precisión (accuracy) ---
# si la accuracy de validación se queda muy por debajo de la de train, hay overfitting
ax2.plot(historial.history['accuracy'], label='accuracy entrenamiento', color='steelblue')  # accuracy en train
ax2.plot(historial.history['val_accuracy'], label='accuracy validación', color='tomato')    # accuracy en test
ax2.set_title('Curva de precisión (Accuracy)')           # título de la gráfica
ax2.set_xlabel('Epoch')                                   # eje x: número de época
ax2.set_ylabel('Accuracy')                                # eje y: porcentaje de aciertos
ax2.legend()                                              # muestro la leyenda

plt.tight_layout()   # ajusto el espaciado para que las dos gráficas no se solapen
plt.show()


In [ ]:
# evalúo el modelo con los datos de test: devuelve la pérdida y la accuracy final
perdida_test, accuracy_test = modelo_red.evaluate(x_test, y_test, verbose=0)
print(f"\nResultados sobre el conjunto de test:")
print(f"  Loss:     {perdida_test:.4f}")   # cuánto se equivoca en promedio
print(f"  Accuracy: {accuracy_test:.4f}")  # porcentaje de predicciones correctas

# genero las predicciones en probabilidades (valores entre 0 y 1)
y_pred_prob = modelo_red.predict(x_test)

# convierto las probabilidades a etiquetas binarias aplicando el umbral de 0.5
# si la probabilidad supera 0.5 → predigo que fue transportado (1), si no → 0
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

# muestro el informe completo de clasificación: precisión, recall y F1 por clase
print("\nInforme de clasificación:")
print(classification_report(y_test, y_pred, target_names=['No transportado', 'Transportado']))


---
### 1️⃣1️⃣ Matriz de confusión

La matriz de confusión muestra de forma visual cuántas predicciones fueron correctas e incorrectas para cada clase:

- **Verdaderos Negativos (TN)**: predije "no transportado" y era correcto ✅
- **Verdaderos Positivos (TP)**: predije "transportado" y era correcto ✅
- **Falsos Positivos (FP)**: predije "transportado" pero no lo estaba ❌
- **Falsos Negativos (FN)**: predije "no transportado" pero sí lo estaba ❌


In [ ]:
# calculo la matriz de confusión comparando las etiquetas reales con las predichas
cm = confusion_matrix(y_test, y_pred)

# visualizo la matriz de confusión con un mapa de calor para facilitar su lectura
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,                          # muestro los valores numéricos dentro de cada celda
    fmt='d',                             # formato entero (sin decimales)
    cmap='Blues',                        # paleta de color azul: más oscuro = más casos
    xticklabels=['No transportado', 'Transportado'],   # etiquetas del eje x (predicciones)
    yticklabels=['No transportado', 'Transportado']    # etiquetas del eje y (valores reales)
)
plt.title('Matriz de Confusión')   # título de la gráfica
plt.ylabel('Valor real')           # eje y: lo que realmente ocurrió
plt.xlabel('Predicción del modelo') # eje x: lo que predijo el modelo
plt.tight_layout()
plt.show()
